https://youtu.be/qrOWFPd5mkE

In [0]:
data = [
    (20124 ,'2020-01-10'),
    (40133 ,'2020-01-15'),
    (65005 ,'2020-01-20'),
    (30005 ,'2020-02-08'),
    (35015 ,'2020-02-19'),
    (15015 ,'2020-03-03'),
    (35035 ,'2020-03-10'),
    (49099 ,'2020-03-14'),
    (84045 ,'2020-03-20'),
    (100106 ,'2020-03-31'),
    (17015 ,'2020-04-04'),
    (36035 ,'2020-04-11'),
    (50099 ,'2020-04-13'),
    (87045 ,'2020-04-22'),
    (101101 ,'2020-04-30'),
    (40015 ,'2020-05-01'),
    (54035 ,'2020-05-09'),
    (71099 ,'2020-05-14'),
    (82045 ,'2020-05-21'),
    (90103 ,'2020-05-25'),
    (99103 ,'2020-05-31'),
    (11015 ,'2020-06-03'),
    (28035 ,'2020-06-10'),
    (38099 ,'2020-06-14'),
    (45045 ,'2020-06-20'),
    (36033 ,'2020-07-09'),
    (40011 ,'2020-07-23'),
    (25001 ,'2020-08-12'),
    (29990 ,'2020-08-26'),
    (20112 ,'2020-09-04'),
    (43991 ,'2020-09-18'),
    (51002 ,'2020-09-29'),
    (26587 ,'2020-10-25'),
    (11000 ,'2020-11-07'),
    (35002 ,'2020-11-16'),
    (56010 ,'2020-11-28'),
    (15099 ,'2020-12-02'),
    (38042 ,'2020-12-11'),
    (73030 ,'2020-12-26')
]

schema = "cases_reported int , dates string"
df = spark.createDataFrame(data= data , schema = schema)
df.show()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
df = (df.withColumn("month", F.month(F.col("dates")))
        .groupby("month")
        .agg(F.sum("cases_reported").alias("cases_reported"))
        .withColumn("Prev_month", F.sum("cases_reported").over(Window.orderBy("month")))
        .withColumn("percentage_increase", 
                    F.round((F.col("cases_reported") / 
                     (F.when((F.col("Prev_month") - F.col("cases_reported"))>0, (F.col("Prev_month") - F.col("cases_reported"))).otherwise(1))
                     *100
                     ),1))
        .withColumn("percentage_increase",F.when(F.col("percentage_increase") == F.col("Prev_month")*100, 0).otherwise(F.col("percentage_increase")))
)
df.display()